# 1. **Modelo de Cadenas de Markov Ocultas (HMM)**

Un **Modelo de Cadena de Markov Oculta (HMM, por sus siglas en inglés)** es una técnica estocástica que se utiliza para modelar procesos generadores de cadenas de observaciones aleatorias y que incluyen elementos ocultos que no se pueden observar directamente. El objetivo del modelo es inferir los estados ocultos a partir de las observaciones disponibles.

### Componentes Fundamentales

Un HMM tiene tres componentes fundamentales: $\pi$, $A$ y $B$. Estos son los parámetros que permiten modelar el comportamiento del sistema, y cómo se pueden generar las observaciones a partir de los estados ocultos. A continuación se explica cada uno:

1. **$\pi$ (Distribución inicial)**: Es un vector que describe la probabilidad inicial de que el sistema se encuentre en cada uno de los posibles states ocultos al comienzo del proceso. $\pi$ establece cómo se distribuyen las probabilidades de los states en el tiempo inicial $t=0$.

2. **$A$ (Matriz de transición)**: Es una tabla que describe las probabilidades de transición entre los states ocultos. Se utiliza para modelar cómo un estado oculto puede pasar a otro en el siguiente paso temporal del proceso.

3. **$B$ (Matriz de emisión)**: Es otra tabla que describe las probabilidades de emisión de observaciones específicas. Estas son las posibles observaciones que podemos esperar ver dadas determinados states ocultos y transiciones entre ellos.

### Ejemplo

Imagine que quieres modelar un proceso en el que un paciente puede estar en uno de tres states ocultos: "Resfriado", "Gripe" o "COVID-19". Al inicio del proceso, el estado del paciente es aleatorio (por ejemplo, con una probabilidad igual de 1/3 para cada uno). Luego, en cada paso temporal, el paciente puede pasar a otro estado con ciertas probabilidades determinadas. Finalmente, hay posibles observaciones que podemos esperar ver: tos, fiebre y dolor de garganta.

### Cálculo de $\alpha_t(i)$

El algoritmo de **Forward** calcula la probabilidad $\alpha_t(i)$ de estar en el estado $i$ en un tiempo temporal dado $t$, cuando hemos observado una secuencia de observaciones hasta ese punto. El cálculo se realiza de manera recursiva utilizando las probabilidades iniciales, la transición y emisión.

### Ejemplo

Supongamos que queremos encontrar $\alpha_2(2)$, es decir, la probabilidad de estar en el estado "COVID-19" en el segundo paso temporal ($t=2$), cuando hemos observado una secuencia de observaciones $O=[o_1, o_2]$ hasta ese momento.

Para este cálculo, se necesita la distribución inicial $\pi$, la transición $A$ y la emisión $B$. Por ejemplo:

$$
\begin{aligned}
\pi = [0.5, 0.3, 0.2] \\
A = \begin{bmatrix}
0.7 & 0.2 & 0.1 \\
0.3 & 0.4 & 0.3 \\
0.2 & 0.3 & 0.5
\end{bmatrix} \\
B = \begin{bmatrix}
0.4 & 0.4 & 0.2 \\
0.5 & 0.3 & 0.2 \\
0.3 & 0.5 & 0.2
\end{bmatrix}
\end{aligned}
$$

Entonces, para calcular $\alpha_2(2)$, podemos seguir los siguientes pasos:

1. En el primer tiempo ($t=0$), la probabilidad de estar en cada estado es igual a $\pi$, por lo que:
$$
\begin{aligned}
\alpha_0(1) &= 0.5 \\
\alpha_0(2) &= 0.3 \\
\alpha_0(3) &= 0.2 
\end{aligned}
$$

2. En el segundo tiempo ($t=1$), las probabilidades de estar en cada estado se calculan utilizando $\alpha_0(i)$ y $A$. Por ejemplo:
$$
\begin{aligned}
\alpha_1(1) &= 0.7 \cdot 0.5 + 0.2 \cdot 0.3 + 0.1 \cdot 0.2 = 0.43 \\
\alpha_1(2) &= 0.3 \cdot 0.5 + 0.4 \cdot 0.3 + 0.3 \cdot 0.2 = 0.33 \\
\alpha_1(3) &= 0.2 \cdot 0.5 + 0.3 \cdot 0.3 + 0.5 \cdot 0.2 = 0.29
\end{aligned}
$$

3. En el tercer tiempo ($t=2$), las probabilidades de estar en cada estado se calculan utilizando $\alpha_1(i)$ y $A$. Por ejemplo:
$$
\begin{aligned}
\alpha_2(1) &= 0.7 \cdot 0.43 + 0.2 \cdot 0.33 + 0.1 \cdot 0.29 = 0.39 \\
\alpha_2(2) &= 0.3 \cdot 0.43 + 0.4 \cdot 0.33 + 0.3 \cdot 0.29 = 0.33 \\
\alpha_2(3) &= 0.2 \cdot 0.43 + 0.3 \cdot 0.33 + 0.5 \cdot 0.29 = 0.33
\end{aligned}
$$

Entonces, $\alpha_2(2)$ es la probabilidad de estar en el estado "COVID-19" en el tercer tiempo, dado las observaciones hasta ese punto y los parámetros del modelo HMM.

In [1]:
import sys

# Add the parent directory to the system path to allow importing from modules located there
sys.path.insert(0, "..")

from likelihood.models.hmm import HMM

In [2]:
# Define the HMM class (the one we just created)
# If you haven't already, make sure to define the HMM class with the methods we discussed.

# Define the parameters of the model
n_states = 3  # Sunny (0), Rainy (1), Cloudy (2)
n_observations = 2  # Walk (0), Shop (1)

# Create an HMM instance
hmm = HMM(n_states, n_observations)

# Generate some synthetic observation sequences (e.g., 3 sequences of 5 days)
# Each number represents an observation: 0 -> Walk, 1 -> Shop
sequences = [
    [0, 1, 0, 0, 1],  # Sequence 1: Walk, Shop, Walk, Walk, Shop
    [1, 1, 0, 1, 0],  # Sequence 2: Shop, Shop, Walk, Shop, Walk
    [0, 0, 0, 1, 1],  # Sequence 3: Walk, Walk, Walk, Shop, Shop
]

# Define the true hidden states for the sequences (ground truth for training/testing)
true_states = [
    [0, 0, 0, 1, 2],  # Sequence 1: Sunny, Sunny, Sunny, Rainy, Cloudy
    [1, 1, 0, 1, 2],  # Sequence 2: Rainy, Rainy, Sunny, Rainy, Cloudy
    [0, 0, 0, 1, 2],  # Sequence 3: Sunny, Sunny, Sunny, Rainy, Cloudy
]

# Test various model configurations
iterations_configs = [20, 50, 100, 200]
best_accuracy = 0
best_config = None

for n_iter in iterations_configs:
    hmm = HMM(n_states, n_observations)
    hmm.baum_welch(sequences, n_iterations=n_iter)
    accuracy = hmm.decoding_accuracy(sequences, true_states)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_config = (n_states, n_observations, n_iter)
        hmm.save_model()

print(f"Best configuration: {best_config} with accuracy {best_accuracy:.2f}%")
print("Pi: ", hmm.pi)
print("A:\n", hmm.A)
print("B:\n", hmm.B)

Best configuration: (3, 2, 200) with accuracy 60.00%
Pi:  [0.35439262 0.24516879 0.40043858]
A:
 [[0.53678304 0.11759566 0.3456213 ]
 [0.24486894 0.60442959 0.15070147]
 [0.25734815 0.25623215 0.4864197 ]]
B:
 [[0.55714163 0.44285837]
 [0.47934544 0.52065456]
 [0.5582844  0.4417156 ]]


In [3]:
hmm = HMM(n_states, n_observations)
hmm = hmm.load_model()
accuracy = hmm.decoding_accuracy(sequences, true_states)
print(f"Decoding Accuracy (model rebuilt): {accuracy:.2f}%")
test_sequence = [0, 1, 0, 0, 1]  # Test sequence: Walk, Shop, Walk, Walk, Shop
predicted_states = hmm.viterbi(test_sequence)
smoothed_probs = hmm.state_probabilities(test_sequence)
total_prob = hmm.sequence_probability(test_sequence)

print(f"Test Sequence: {test_sequence}")
print(f"Predicted States: {predicted_states}")
print(f"Probabilities of hidden states: {smoothed_probs}")
print(f"Total probability: {total_prob}")

Decoding Accuracy (model rebuilt): 60.00%
Test Sequence: [0, 1, 0, 0, 1]
Predicted States: [0 0 0 0 0]
Probabilities of hidden states: [[0.28873463 0.4565849  0.25468047]
 [0.33941198 0.33264173 0.32794629]
 [0.30965021 0.36871896 0.32163083]
 [0.28682214 0.40581216 0.3073657 ]
 [0.32725088 0.33114961 0.34159951]]
Total probability: [0.32725088 0.33114961 0.34159951]
